# ReviewRadar: 3-Class Customer Review Sentiment Classification
**Course:** AI and Expert Systems  
**Task:** 3-Class Sentiment Classification (`Negative`, `Neutral`, `Positive`)  
**Datasets:** Amazon Reviews 2023 & Yelp Open Dataset (pre-downloaded records in Google Drive)  
**Models:**  
1. **Baseline Model:** TF-IDF + Logistic Regression  
2. **Dense Semantic Embeddings Model:** Pretrained `all-MiniLM-L6-v2` Embeddings + Logistic Regression  

---
> **Important Instructions for Running in Google Colab:**
> - **Run All Cells in Order:** Use **Runtime** &rarr; **Run all** (or run each cell from top to bottom).
> - **Runtime:** Use GPU for faster sentence-transformer embeddings (**Runtime** &rarr; **Change runtime type** &rarr; select **T4 GPU**).
> - **Data Location:** Both dataset files (~100k records each) are already downloaded into your Google Drive at: `/content/drive/MyDrive/reviewradar/datasets/`.
> - **No Re-downloads:** This notebook strictly operates on your existing files (`amazon_100k_raw.csv` / `yelp_100k_raw.csv` or JSONL equivalents).
> - **Memory Optimization:** Files are ingested in manageable chunks (`chunksize=25,000`) to prevent high RAM consumption.
> - **Pretrained Feature Extraction:** The transformer model (`all-MiniLM-L6-v2`) is used as a **frozen embedding extractor** to generate dense vectors. It is **not fine-tuned**.

## 1. Setup Environment & Mount Google Drive
In this step, we install `sentence-transformers` to obtain pretrained dense sentence embeddings, import required data science libraries, set a fixed random seed (42), mount your Google Drive, and initialize project directory paths.

In [ ]:
# Install sentence-transformers and ensure required libraries are present
!pip install -q sentence-transformers joblib scikit-learn matplotlib seaborn

import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sentence_transformers import SentenceTransformer

# Set fixed random seed for reproducible results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Initialize default Google Drive directories & dataset paths right away
PROJECT_DIR = '/content/drive/MyDrive/reviewradar'
DATA_DIR = os.path.join(PROJECT_DIR, 'datasets')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

AMAZON_FILE = os.path.join(DATA_DIR, 'amazon_100k_raw.csv')
YELP_FILE = os.path.join(DATA_DIR, 'yelp_100k_raw.csv')

print("✓ Environment initialized and Google Drive mounted successfully.")

## 2. Configure Directory Paths & Inspect Dataset Files
We set the project path to `/content/drive/MyDrive/reviewradar` and the dataset path to `/content/drive/MyDrive/reviewradar/datasets`.

The cell below lists every file inside your `datasets/` directory so you can confirm the exact filenames.

In [ ]:
# Define Google Drive directories
PROJECT_DIR = '/content/drive/MyDrive/reviewradar'
DATA_DIR = os.path.join(PROJECT_DIR, 'datasets')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')

# Create the models directory in Google Drive if it doesn't already exist
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Project Directory : {PROJECT_DIR}")
print(f"Datasets Directory: {DATA_DIR}")
print(f"Models Directory  : {MODELS_DIR}")

# Inspect files present in the datasets folder
if os.path.exists(DATA_DIR):
    files_in_datasets = os.listdir(DATA_DIR)
    print(f"\nFiles detected in '{DATA_DIR}':")
    for f in sorted(files_in_datasets):
        file_full_path = os.path.join(DATA_DIR, f)
        if os.path.isfile(file_full_path):
            size_mb = os.path.getsize(file_full_path) / (1024 * 1024)
            print(f"  • {f} ({size_mb:.2f} MB)")
else:
    print(f"\n[WARNING] Directory does not exist: {DATA_DIR}")
    print("Please verify your Google Drive folder structure: MyDrive/reviewradar/datasets/")

### Automatic Yelp ZIP Extraction (If Applicable)
If your Yelp dataset is already an uncompressed `.csv` or `.json`/`.jsonl` file (such as `yelp_100k_raw.csv`), extraction is skipped automatically.
If only a `.zip` archive is present in `datasets/`, the code below extracts the review file to Colab's local scratch disk (`/content/yelp_extracted/`).

In [ ]:
LOCAL_YELP_DIR = '/content/yelp_extracted'
os.makedirs(LOCAL_YELP_DIR, exist_ok=True)

def check_and_extract_yelp_zip(data_dir):
    """
    If an uncompressed Yelp CSV/JSON/JSONL file exists in data_dir, returns its path.
    Otherwise, if a Yelp ZIP archive is found, extracts the review file to /content/yelp_extracted/.
    """
    if not os.path.exists(data_dir):
        return None

    # 1. Check if an uncompressed review csv/json/jsonl file already exists in datasets/
    for fname in sorted(os.listdir(data_dir)):
        fn_lower = fname.lower()
        if 'yelp' in fn_lower and fn_lower.endswith(('.csv', '.json', '.jsonl', '.jsonl.gz', '.csv.gz')):
            print(f"Found uncompressed Yelp file in datasets: {fname}")
            return os.path.join(data_dir, fname)

    # 2. Check for a ZIP archive
    for fname in sorted(os.listdir(data_dir)):
        if fname.lower().endswith('.zip') and 'yelp' in fname.lower():
            zip_path = os.path.join(data_dir, fname)
            print(f"Detected Yelp ZIP file: {zip_path}")
            with zipfile.ZipFile(zip_path, 'r') as z:
                review_member = None
                for member in z.namelist():
                    m_lower = member.lower()
                    if 'review' in m_lower and m_lower.endswith(('.json', '.jsonl', '.csv')):
                        review_member = member
                        break

                if review_member:
                    print(f"Extracting '{review_member}' to local scratch ({LOCAL_YELP_DIR})...")
                    z.extract(review_member, LOCAL_YELP_DIR)
                    extracted_path = os.path.join(LOCAL_YELP_DIR, review_member)
                    print(f"✓ Extraction complete: {extracted_path}")
                    return extracted_path
                else:
                    print("Could not find a review file inside the ZIP archive.")
    return None

extracted_yelp_path = check_and_extract_yelp_zip(DATA_DIR)

### Set Dataset File Paths (`AMAZON_FILE` and `YELP_FILE`)
Below, we assign `AMAZON_FILE` and `YELP_FILE` (`amazon_100k_raw.csv` and `yelp_100k_raw.csv` by default, with automatic fallback discovery if your filenames differ).

In [ ]:
def resolve_dataset_paths(data_dir, fallback_yelp=None):
    """
    Resolves AMAZON_FILE and YELP_FILE from data_dir.
    Checks default names first (amazon_100k_raw.csv, yelp_100k_raw.csv),
    then scans data_dir for any matching Amazon/Yelp CSV or JSONL files.
    """
    amazon_path = os.path.join(data_dir, 'amazon_100k_raw.csv')
    yelp_path = os.path.join(data_dir, 'yelp_100k_raw.csv')

    if os.path.exists(data_dir):
        files = sorted(os.listdir(data_dir))
        if not os.path.exists(amazon_path):
            for f in files:
                fl = f.lower()
                if 'amazon' in fl and fl.endswith(('.csv', '.jsonl', '.json', '.jsonl.gz', '.csv.gz')):
                    amazon_path = os.path.join(data_dir, f)
                    break
        if not os.path.exists(yelp_path):
            if fallback_yelp and os.path.exists(fallback_yelp):
                yelp_path = fallback_yelp
            else:
                for f in files:
                    fl = f.lower()
                    if 'yelp' in fl and fl.endswith(('.csv', '.jsonl', '.json', '.jsonl.gz', '.csv.gz')):
                        yelp_path = os.path.join(data_dir, f)
                        break
    return amazon_path, yelp_path

AMAZON_FILE, YELP_FILE = resolve_dataset_paths(
    DATA_DIR,
    fallback_yelp=globals().get('extracted_yelp_path')
)

print(f"AMAZON_FILE : {AMAZON_FILE}")
print(f"YELP_FILE   : {YELP_FILE}")

# Verification assertions to avoid silent errors
assert AMAZON_FILE and os.path.exists(AMAZON_FILE), (
    f"Amazon file not found: {AMAZON_FILE}. Please check your datasets/ folder."
)
assert YELP_FILE and os.path.exists(YELP_FILE), (
    f"Yelp file not found: {YELP_FILE}. Please check your datasets/ folder."
)
print("✓ Both dataset files exist and are ready for processing.")

## 3. Data Ingestion & Preprocessing

### Cleaning & Labeling Rules:
1. **Chunked Reading**: To avoid Colab RAM exhaustion, files are read in chunks (`chunksize=25,000`). Both `.csv` and `.json`/`.jsonl` formats are supported automatically.
2. **Column Inspection**: Check that expected fields are present (`rating`, `title`, `text` for Amazon; `stars`, `text` for Yelp) and show clear errors if missing.
3. **Text Formulation**:
   - **Amazon**: Combine `title` and `text` into `review_text` (e.g., `"Title. Body"`). If title is missing or empty, use body text.
   - **Yelp**: Use `text` as `review_text`.
4. **Filtering**:
   - Remove records with ratings outside 1 to 5.
   - Remove records with empty or whitespace-only `review_text`.
5. **Whitespace Normalization & Negation Preservation**:
   - Trim leading/trailing whitespace and normalize internal whitespace.
   - **Do NOT remove negation words** (`not`, `no`, `never`, `neither`, `cannot`) as they dictate sentiment polarity.
6. **Deduplication**:
   - Drop exact duplicate `review_text` within each dataset.
7. **3-Class Sentiment Mapping**:
   - Ratings **1 or 2** &rarr; **`Negative`**
   - Rating **3** &rarr; **`Neutral`**
   - Ratings **4 or 5** &rarr; **`Positive`**
8. **Selected Output Columns**: `['review_text', 'rating', 'sentiment', 'source']`.

In [ ]:
def map_rating_to_sentiment(rating):
    """
    Maps 1-5 numerical ratings to 3-class sentiment.
    1, 2 -> Negative
    3    -> Neutral
    4, 5 -> Positive
    """
    if rating in [1, 2, 1.0, 2.0]:
        return 'Negative'
    elif rating in [3, 3.0]:
        return 'Neutral'
    elif rating in [4, 5, 4.0, 5.0]:
        return 'Positive'
    return None

def normalize_text_series(series):
    """
    Normalizes whitespace while preserving all negation words and vocabulary.
    """
    return series.fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

def get_chunk_reader(file_path, chunksize=25000):
    """
    Returns a chunked pandas reader supporting both CSV (.csv, .csv.gz)
    and JSON-Lines (.json, .jsonl, .jsonl.gz) files automatically.
    """
    fp_lower = file_path.lower()
    if fp_lower.endswith(('.csv', '.csv.gz')):
        return pd.read_csv(
            file_path,
            chunksize=chunksize,
            encoding='utf-8',
            on_bad_lines='skip',
            low_memory=False
        )
    else:
        return pd.read_json(
            file_path,
            lines=True,
            chunksize=chunksize,
            compression='infer'
        )

In [ ]:
# Ensure AMAZON_FILE is defined even if earlier path cells were skipped
if 'AMAZON_FILE' not in globals() or not AMAZON_FILE:
    PROJECT_DIR = globals().get('PROJECT_DIR', '/content/drive/MyDrive/reviewradar')
    DATA_DIR = globals().get('DATA_DIR', os.path.join(PROJECT_DIR, 'datasets'))
    AMAZON_FILE = os.path.join(DATA_DIR, 'amazon_100k_raw.csv')

def process_amazon_dataset(file_path, chunksize=25000):
    """
    Streams Amazon Reviews in chunks (CSV or JSONL), verifies columns, cleans records,
    combines title + text, and maps sentiment.
    """
    print(f"\nProcessing Amazon dataset: {file_path}")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Amazon dataset file not found at: {file_path}")

    raw_count = 0
    invalid_rating_count = 0
    empty_text_count = 0
    cleaned_chunks = []
    first_chunk = True

    reader = get_chunk_reader(file_path, chunksize=chunksize)

    for chunk in reader:
        raw_count += len(chunk)

        # Normalize column names if needed (strip whitespace)
        chunk.columns = [str(c).strip() for c in chunk.columns]

        # Verify columns on first chunk
        if first_chunk:
            print("Amazon detected columns:", list(chunk.columns))
            if 'rating' not in chunk.columns or 'text' not in chunk.columns:
                raise KeyError(
                    f"Amazon dataset missing required columns ('rating' or 'text'). Found: {list(chunk.columns)}"
                )
            preview_cols = [c for c in ['rating', 'title', 'text'] if c in chunk.columns]
            sample_row = chunk[preview_cols].head(1).to_dict(orient='records')
            print("Amazon sample row preview:", sample_row)
            first_chunk = False

        # 1. Filter valid ratings [1, 5]
        chunk['rating_num'] = pd.to_numeric(chunk['rating'], errors='coerce')
        valid_rating_mask = chunk['rating_num'].isin([1, 2, 3, 4, 5])
        invalid_rating_count += int((~valid_rating_mask).sum())
        chunk = chunk[valid_rating_mask].copy()

        # 2. Combine title and text into review_text
        has_title = 'title' in chunk.columns
        title_cleaned = normalize_text_series(chunk['title']) if has_title else ''
        body_cleaned = normalize_text_series(chunk['text'])

        if has_title:
            combined = np.where(
                title_cleaned.str.len() > 0,
                title_cleaned + '. ' + body_cleaned,
                body_cleaned
            )
            chunk['review_text'] = pd.Series(combined, index=chunk.index).str.strip()
        else:
            chunk['review_text'] = body_cleaned

        # 3. Filter empty text
        empty_mask = chunk['review_text'].str.len() == 0
        empty_text_count += int(empty_mask.sum())
        chunk = chunk[~empty_mask].copy()

        # 4. Map sentiment label
        chunk['rating'] = chunk['rating_num'].astype(int)
        chunk['sentiment'] = chunk['rating'].map(map_rating_to_sentiment)
        chunk['source'] = 'Amazon'

        cleaned_chunks.append(chunk[['review_text', 'rating', 'sentiment', 'source']])

    df_amazon = pd.concat(cleaned_chunks, ignore_index=True) if cleaned_chunks else pd.DataFrame()

    # 5. Deduplication within Amazon dataset
    count_before_dedup = len(df_amazon)
    df_amazon = df_amazon.drop_duplicates(subset=['review_text']).reset_index(drop=True)
    duplicates_removed = int(count_before_dedup - len(df_amazon))

    audit_stats = {
        'source': 'Amazon',
        'raw_count': int(raw_count),
        'invalid_rating': int(invalid_rating_count),
        'empty_text': int(empty_text_count),
        'duplicates_removed': int(duplicates_removed),
        'final_usable': int(len(df_amazon))
    }

    print(f"✓ Amazon cleaning finished: {len(df_amazon):,} usable records.")
    return df_amazon, audit_stats

df_amazon, amazon_audit = process_amazon_dataset(AMAZON_FILE)

In [ ]:
# Ensure YELP_FILE is defined even if earlier path cells were skipped
if 'YELP_FILE' not in globals() or not YELP_FILE:
    PROJECT_DIR = globals().get('PROJECT_DIR', '/content/drive/MyDrive/reviewradar')
    DATA_DIR = globals().get('DATA_DIR', os.path.join(PROJECT_DIR, 'datasets'))
    YELP_FILE = os.path.join(DATA_DIR, 'yelp_100k_raw.csv')

def process_yelp_dataset(file_path, chunksize=25000):
    """
    Streams Yelp Reviews in chunks (CSV or JSONL), verifies columns, cleans records,
    extracts text, and maps sentiment.
    """
    print(f"\nProcessing Yelp dataset: {file_path}")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Yelp dataset file not found at: {file_path}")

    raw_count = 0
    invalid_rating_count = 0
    empty_text_count = 0
    cleaned_chunks = []
    first_chunk = True

    reader = get_chunk_reader(file_path, chunksize=chunksize)

    for chunk in reader:
        raw_count += len(chunk)

        # Normalize column names if needed
        chunk.columns = [str(c).strip() for c in chunk.columns]
        rating_col = 'stars' if 'stars' in chunk.columns else ('rating' if 'rating' in chunk.columns else None)

        # Verify columns on first chunk
        if first_chunk:
            print("Yelp detected columns:", list(chunk.columns))
            if rating_col is None or 'text' not in chunk.columns:
                raise KeyError(
                    f"Yelp dataset missing required columns ('stars' and 'text'). Found: {list(chunk.columns)}"
                )
            sample_row = chunk[[rating_col, 'text']].head(1).to_dict(orient='records')
            print("Yelp sample row preview:", sample_row)
            first_chunk = False

        # 1. Filter valid ratings [1, 5]
        chunk['stars_num'] = pd.to_numeric(chunk[rating_col], errors='coerce')
        valid_rating_mask = chunk['stars_num'].isin([1, 2, 3, 4, 5])
        invalid_rating_count += int((~valid_rating_mask).sum())
        chunk = chunk[valid_rating_mask].copy()

        # 2. Extract and normalize review_text
        chunk['review_text'] = normalize_text_series(chunk['text'])

        # 3. Filter empty text
        empty_mask = chunk['review_text'].str.len() == 0
        empty_text_count += int(empty_mask.sum())
        chunk = chunk[~empty_mask].copy()

        # 4. Map sentiment label
        chunk['rating'] = chunk['stars_num'].astype(int)
        chunk['sentiment'] = chunk['rating'].map(map_rating_to_sentiment)
        chunk['source'] = 'Yelp'

        cleaned_chunks.append(chunk[['review_text', 'rating', 'sentiment', 'source']])

    df_yelp = pd.concat(cleaned_chunks, ignore_index=True) if cleaned_chunks else pd.DataFrame()

    # 5. Deduplication within Yelp dataset
    count_before_dedup = len(df_yelp)
    df_yelp = df_yelp.drop_duplicates(subset=['review_text']).reset_index(drop=True)
    duplicates_removed = int(count_before_dedup - len(df_yelp))

    audit_stats = {
        'source': 'Yelp',
        'raw_count': int(raw_count),
        'invalid_rating': int(invalid_rating_count),
        'empty_text': int(empty_text_count),
        'duplicates_removed': int(duplicates_removed),
        'final_usable': int(len(df_yelp))
    }

    print(f"✓ Yelp cleaning finished: {len(df_yelp):,} usable records.")
    return df_yelp, audit_stats

df_yelp, yelp_audit = process_yelp_dataset(YELP_FILE)

## 4. Data Quality Audit & Exploratory Data Analysis (EDA)
Here we report the actual verified numbers found by our preprocessing pipeline:
1. **Cleaning Audit**: Raw records read, invalid ratings dropped, empty texts dropped, duplicates removed, and final usable records.
2. **Sentiment Distribution**: Class counts and percentages for Amazon, Yelp, and Combined.
3. **Visualizations**: Sentiment class distribution comparison and review text length boxplots.

In [ ]:
# Build data audit summary table
audit_df = pd.DataFrame([amazon_audit, yelp_audit])
audit_df['retention_rate_%'] = ((audit_df['final_usable'] / audit_df['raw_count']) * 100).round(2)

print("=" * 75)
print("DATA QUALITY & CLEANING AUDIT REPORT")
print("=" * 75)
display(audit_df)

# Concatenate into combined dataset
df_all = pd.concat([df_amazon, df_yelp], ignore_index=True)
print(f"\nCombined usable reviews: {len(df_all):,}")

In [ ]:
def display_sentiment_breakdown(df, source_label):
    counts = df['sentiment'].value_counts()
    percentages = (df['sentiment'].value_counts(normalize=True) * 100).round(2)
    summary = pd.DataFrame({'Review Count': counts, 'Percentage (%)': percentages})
    summary.loc['Total'] = [len(df), 100.0]
    print(f"\n--- Sentiment Breakdown: {source_label} ---")
    display(summary)
    return summary

amazon_breakdown = display_sentiment_breakdown(df_amazon, "Amazon Reviews")
yelp_breakdown = display_sentiment_breakdown(df_yelp, "Yelp Reviews")
combined_breakdown = display_sentiment_breakdown(df_all, "Combined Dataset (Amazon + Yelp)")

In [ ]:
# Exploratory Visualizations
df_all['word_count'] = df_all['review_text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Sentiment Class Distribution by Source
sns.countplot(
    data=df_all,
    x='sentiment',
    hue='source',
    order=['Negative', 'Neutral', 'Positive'],
    palette={'Amazon': '#ff9900', 'Yelp': '#d32323'},
    ax=axes[0]
)
axes[0].set_title('Sentiment Distribution by Source (Amazon vs. Yelp)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Review Count')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# 2. Review Word Count Distribution (capped at 99th percentile for clean display)
cap_words = df_all['word_count'].quantile(0.99)
sns.boxplot(
    data=df_all[df_all['word_count'] <= cap_words],
    x='sentiment',
    y='word_count',
    hue='sentiment',
    order=['Negative', 'Neutral', 'Positive'],
    palette='Set2',
    legend=False,
    ax=axes[1]
)
axes[1].set_title(f'Word Count Distribution by Sentiment (<= {int(cap_words)} words)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment Class')
axes[1].set_ylabel('Number of Words')
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 5. Stratified Train / Test Split (80% / 20%)

To ensure that both data sources (`Amazon`, `Yelp`) and all three sentiment classes (`Negative`, `Neutral`, `Positive`) are distributed in exact proportions between train and test sets, we stratify by a combined composite key:
$$\text{strat\_key} = \text{source} + \text{'_'} + \text{sentiment}$$

Both Model 1 and Model 2 will be trained and evaluated on **this identical split** to ensure a fair comparison.

In [ ]:
# Build compound stratification column (e.g. Amazon_Positive, Yelp_Negative)
df_all['strat_col'] = df_all['source'] + '_' + df_all['sentiment']

# 80% Train, 20% Test split with fixed random seed (42)
train_df, test_df = train_test_split(
    df_all,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=df_all['strat_col']
)

X_train_text = train_df['review_text'].values
y_train = train_df['sentiment'].values

X_test_text = test_df['review_text'].values
y_test = test_df['sentiment'].values

print(f"Total Usable Records : {len(df_all):,}")
print(f"Training Set (80%)   : {len(train_df):,} records")
print(f"Testing Set (20%)    : {len(test_df):,} records")

# Verify train/test sentiment alignment
train_prop = (train_df['sentiment'].value_counts(normalize=True) * 100).round(2).to_dict()
test_prop  = (test_df['sentiment'].value_counts(normalize=True) * 100).round(2).to_dict()
print(f"\nTrain Sentiment Proportions (%): {train_prop}")
print(f"Test Sentiment Proportions  (%): {test_prop}")

## 6. Model 1 - Baseline: TF-IDF + Logistic Regression
- **Feature Extraction**: `TfidfVectorizer` computes n-gram word statistics (unigrams & bigrams, max 10,000 features).
- **Classifier**: `LogisticRegression` with `class_weight='balanced'` to offset class imbalance.
- **Data Leakage Protection**: Handled via a scikit-learn `Pipeline`, fitting vectorizer parameters strictly on training text.

In [ ]:
print("Training Model 1: TF-IDF + Logistic Regression Baseline...")

# Construct scikit-learn Pipeline
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        lowercase=True,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_SEED,
        solver='lbfgs'
    ))
])

# Fit on training data
tfidf_pipeline.fit(X_train_text, y_train)

# Predict on test set
y_pred_tfidf = tfidf_pipeline.predict(X_test_text)

print("✓ Model 1 training and test inference complete.")

## 7. Model 2 - Actual Dense Text Embeddings: `all-MiniLM-L6-v2` + Logistic Regression

> **Key Concept: Sparse vs. Dense Representations**
> - **TF-IDF (Model 1)** produces *sparse high-dimensional vectors* based solely on exact token frequencies. It misses synonymy and deep semantic context.
> - **MiniLM (Model 2)** produces *dense 384-dimensional continuous vectors* encoding deep semantic and contextual relationships learned from billions of sentences.
> - **Transformer Usage:** We use the pretrained `all-MiniLM-L6-v2` encoder strictly as a **frozen feature extractor**. We do **not** fine-tune the transformer weights; a Logistic Regression classifier is trained directly on the dense embedding vectors.

In [ ]:
# Select hardware device (GPU if available)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Hardware device for SentenceTransformer: {device.upper()}")
if device == 'cpu':
    print("Note: Running on CPU. For faster encoding, enable GPU in Colab (Runtime -> Change runtime type).")

# Load pretrained sentence-transformers model
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

print(f"✓ Loaded pretrained encoder: {EMBEDDING_MODEL_NAME}")
print(f"  Embedding dimension: 384")

In [ ]:
# Encode review texts into dense embeddings in batches with a progress bar
BATCH_SIZE = 64

print("Generating dense embeddings for TRAINING reviews...")
X_train_embeddings = embedding_model.encode(
    X_train_text.tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nGenerating dense embeddings for TESTING reviews...")
X_test_embeddings = embedding_model.encode(
    X_test_text.tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"\n✓ Training embeddings shape: {X_train_embeddings.shape}")
print(f"✓ Testing embeddings shape : {X_test_embeddings.shape}")

In [ ]:
print("Training Model 2 Classifier: Logistic Regression on Dense Embeddings...")

embedding_classifier = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_SEED,
    solver='lbfgs'
)

# Train classifier on dense embedding vectors
embedding_classifier.fit(X_train_embeddings, y_train)

# Predict on test embedding vectors
y_pred_embedding = embedding_classifier.predict(X_test_embeddings)

print("✓ Model 2 training and test inference complete.")

## 8. Evaluation & Comparative Analysis
We evaluate both models on the test set across standard classification metrics:
- **Accuracy**
- **Macro Precision, Recall, and F1-Score** (treats all 3 classes equally, highlighting performance on minority classes)
- **Weighted F1-Score** (accounts for class imbalance)
- **Confusion Matrices** & **Per-Class Classification Reports**

All metrics are calculated dynamically from actual predictions (no hardcoded numbers).

In [ ]:
CLASSES = ['Negative', 'Neutral', 'Positive']

def compute_metrics(y_true, y_pred, model_title):
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', labels=CLASSES, zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', labels=CLASSES, zero_division=0
    )
    return {
        'Model': model_title,
        'Accuracy': float(acc),
        'Macro Precision': float(macro_p),
        'Macro Recall': float(macro_r),
        'Macro F1': float(macro_f1),
        'Weighted F1': float(weighted_f1)
    }

m1_metrics = compute_metrics(y_test, y_pred_tfidf, "Model 1: TF-IDF + Logistic Regression")
m2_metrics = compute_metrics(y_test, y_pred_embedding, "Model 2: all-MiniLM-L6-v2 + Logistic Regression")

comparison_table = pd.DataFrame([m1_metrics, m2_metrics]).set_index('Model')

print("=" * 75)
print("MODEL EVALUATION COMPARISON TABLE")
print("=" * 75)
display(comparison_table.round(4))

In [ ]:
print("=" * 65)
print("CLASSIFICATION REPORT: MODEL 1 (TF-IDF + Logistic Regression)")
print("=" * 65)
print(classification_report(y_test, y_pred_tfidf, labels=CLASSES, digits=4))

print("=" * 65)
print("CLASSIFICATION REPORT: MODEL 2 (all-MiniLM-L6-v2 + Logistic Regression)")
print("=" * 65)
print(classification_report(y_test, y_pred_embedding, labels=CLASSES, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Model 1 Confusion Matrix Heatmap
cm1 = confusion_matrix(y_test, y_pred_tfidf, labels=CLASSES)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Model 1: TF-IDF + LR\nConfusion Matrix", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# Model 2 Confusion Matrix Heatmap
cm2 = confusion_matrix(y_test, y_pred_embedding, labels=CLASSES)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title("Model 2: MiniLM Embeddings + LR\nConfusion Matrix", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

# Comparison Grouped Bar Chart
metrics_plot_df = comparison_table[['Accuracy', 'Macro F1', 'Weighted F1']].reset_index().melt(
    id_vars='Model', var_name='Metric', value_name='Score'
)
sns.barplot(data=metrics_plot_df, x='Metric', y='Score', hue='Model', palette=['#3b82f6', '#10b981'], ax=axes[2])
axes[2].set_title("Performance Comparison\n(Accuracy, Macro F1, Weighted F1)", fontsize=11, fontweight='bold')
axes[2].set_ylim(0.0, 1.05)
axes[2].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 9. Save Trained Models & Metadata to Google Drive

We save the trained pipelines and evaluation artifacts to `/content/drive/MyDrive/reviewradar/models/` for seamless deployment in our local FastAPI backend:
1. **`tfidf_pipeline.joblib`**: Full scikit-learn Pipeline containing both fitted `TfidfVectorizer` and `LogisticRegression`.
2. **`embedding_classifier.joblib`**: Trained `LogisticRegression` classifier for `all-MiniLM-L6-v2` embeddings.
3. **`metrics.json`**: Complete metrics, dataset counts, and class distributions.
4. **`model_info.json`**: Pretrained model name, class order, and dimensions for inference.

In [ ]:
# Ensure MODELS_DIR exists
if 'MODELS_DIR' not in globals() or not MODELS_DIR:
    PROJECT_DIR = globals().get('PROJECT_DIR', '/content/drive/MyDrive/reviewradar')
    MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Target paths inside Google Drive
TFIDF_PIPELINE_PATH = os.path.join(MODELS_DIR, 'tfidf_pipeline.joblib')
EMBEDDING_CLASSIFIER_PATH = os.path.join(MODELS_DIR, 'embedding_classifier.joblib')
METRICS_PATH = os.path.join(MODELS_DIR, 'metrics.json')
MODEL_INFO_PATH = os.path.join(MODELS_DIR, 'model_info.json')

# 1. Save TF-IDF Pipeline
joblib.dump(tfidf_pipeline, TFIDF_PIPELINE_PATH)

# 2. Save Embedding Classifier
joblib.dump(embedding_classifier, EMBEDDING_CLASSIFIER_PATH)

# 3. Save Metrics JSON (converting all counts to native Python int for JSON safety)
metrics_data = {
    'dataset_statistics': {
        'amazon_raw_inspected': int(amazon_audit['raw_count']),
        'amazon_usable': int(amazon_audit['final_usable']),
        'yelp_raw_inspected': int(yelp_audit['raw_count']),
        'yelp_usable': int(yelp_audit['final_usable']),
        'total_usable_reviews': int(len(df_all)),
        'train_samples_80pct': int(len(train_df)),
        'test_samples_20pct': int(len(test_df))
    },
    'sentiment_distribution': {
        'combined': {str(k): int(v) for k, v in df_all['sentiment'].value_counts().items()},
        'train': {str(k): int(v) for k, v in train_df['sentiment'].value_counts().items()},
        'test': {str(k): int(v) for k, v in test_df['sentiment'].value_counts().items()}
    },
    'evaluation_results': {
        'model_1_tfidf_lr': m1_metrics,
        'model_2_minilm_lr': m2_metrics
    }
}
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_data, f, indent=2)

# 4. Save Model Info JSON
model_info_data = {
    'project': 'ReviewRadar',
    'task': '3-class sentiment classification',
    'classes': CLASSES,
    'embedding_model_name': EMBEDDING_MODEL_NAME,
    'embedding_dimension': 384,
    'random_seed': int(RANDOM_SEED),
    'artifacts': {
        'tfidf_pipeline': 'tfidf_pipeline.joblib',
        'embedding_classifier': 'embedding_classifier.joblib'
    }
}
with open(MODEL_INFO_PATH, 'w', encoding='utf-8') as f:
    json.dump(model_info_data, f, indent=2)

# Verification: Check existence and file size of saved artifacts
print("=" * 75)
print("VERIFYING SAVED ARTIFACTS IN GOOGLE DRIVE")
print("=" * 75)
artifacts = [TFIDF_PIPELINE_PATH, EMBEDDING_CLASSIFIER_PATH, METRICS_PATH, MODEL_INFO_PATH]
for path in artifacts:
    exists = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    status = f"✓ OK ({size_kb:.1f} KB)" if exists else "✗ NOT FOUND"
    print(f"{os.path.basename(path):<30} : {status}")
    print(f"  Full Path: {path}")

## 10. End-to-End Pipeline Summary & Limitations

### Pipeline Recap
1. **Pre-downloaded Subsets**: Analyzed ~100k reviews from Amazon Reviews 2023 and ~100k reviews from the Yelp Open Dataset.
2. **Streaming & Quality Control**: Processed data in memory-safe chunks; validated star ratings (1–5); merged Amazon title and body text; normalized whitespace while preserving negation cues (`not`, `no`, `never`); and removed exact duplicate review strings.
3. **3-Class Labeling**: Mapped 1–2 stars to `Negative`, 3 stars to `Neutral`, and 4–5 stars to `Positive`.
4. **Stratified Split**: 80/20 train/test partition stratified by both dataset source and sentiment class.
5. **Dual Modeling**:
   - **Model 1**: TF-IDF (10,000 unigram/bigram features) with balanced Logistic Regression.
   - **Model 2**: Pretrained `all-MiniLM-L6-v2` dense embeddings (384-dimensional) with balanced Logistic Regression.
6. **Artifact Export**: Stored model pipelines, classification metrics, and inference configurations in Google Drive.

### Project Limitations & Critical Discussion
- **Feature Extraction vs. Fine-Tuning**: `all-MiniLM-L6-v2` was applied strictly as a **frozen feature extractor**. The transformer weights were not updated through backpropagation on the review datasets. Full fine-tuning (e.g., updating attention weights with an end-to-end classification head) could adapt representations further to informal colloquial slang, but requires substantially higher GPU memory and training time.
- **Cross-Platform Domain Shift**: Amazon reviews (consumer electronics, household items, packaging) and Yelp reviews (restaurants, local service staff, wait times) exhibit distinct vocabularies and tones. Training a single joint classifier tests generalizability, but cross-domain variance remains.
- **Neutral Class Ambiguity**: Customer reviews with 3 stars frequently display mixed sentiment (e.g., *"Great food but terrible service"*). Consequently, `Neutral` precision and recall are typically lower across both models than `Positive` or `Negative`.
- **Inference Latency Trade-Off**: Model 1 (TF-IDF) is extremely fast on CPU-only machines. Model 2 (MiniLM) requires generating 384-dimensional dense vectors via transformer matrix operations, introducing slightly higher latency per request during live backend inference.